# 07b — Hybrid Integration: Lexical-Graph Indexing & Query

1. Load documents from Part 1
2. Index into lexical-graph (Neptune + OpenSearch)
3. Query with semantic search
4. Correlate results back to document-graph (lineage)


In [ ]:
# Setup
import json, os, pickle, re
from pathlib import Path
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory, VectorStoreFactory
from graphrag_toolkit.lexical_graph import LexicalGraphIndex, LexicalGraphQueryEngine

GRAPH_STORE = 'neptune-db://obs-app-dev-graph.cluster-czupj1ab2tc6.us-east-1.neptune.amazonaws.com:8182'
VECTOR_STORE = 'aoss://https://1lci0mi6xy7hpex8fr0b.us-east-1.aoss.amazonaws.com'
TENANT = 'hybrid_demo'

graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE)
vector_store = VectorStoreFactory.for_vector_store(VECTOR_STORE)
print('Stores connected')


## Step 1: Load documents from Part 1


In [ ]:
foundation_dir = Path('output/hybrid_foundation')

with open(foundation_dir / 'lexical_documents.pkl', 'rb') as f:
    lexical_documents = pickle.load(f)

print(f'Loaded {len(lexical_documents)} documents')
print(f'Sample: {lexical_documents[0].text[:150]}...')


## Step 2: Index into lexical-graph


In [ ]:
# Index into lexical-graph (batch writes enabled — Neptune 1.4.7.0 supports it)
print(f"Indexing {len(lexical_documents)} documents (batch writes ON)...")

graph_index = LexicalGraphIndex(graph_store, vector_store)
graph_index.extract_and_build(lexical_documents, show_progress=True)

print("Indexing complete")


## Step 3: Query with semantic search


In [ ]:
query_engine = LexicalGraphQueryEngine.for_traversal_based_search(graph_store, vector_store)

queries = [
    'What research discusses machine learning?',
    'Who are the authors working on graph databases?',
    'What citations exist between papers?',
]

for q in queries:
    print(f'\nQuery: {q}')
    results = query_engine.retrieve(q)
    if isinstance(results, list):
        print(f'  Results: {len(results)}')
        for i, r in enumerate(results[:2], 1):
            text = getattr(r.node, 'text', str(r))[:150]
            print(f'  {i}. {text}...')
    else:
        print(f'  Response: {str(results)[:150]}')


## Step 4: Correlate back to document-graph (lineage)


In [ ]:
# Correlate: query results → Source nodes → document-graph
gs_direct = GraphStoreFactory.for_graph_store(GRAPH_STORE).__enter__()

q = 'What research discusses machine learning?'
results = query_engine.retrieve(q)
print(f'Query: {q}')
print(f'Results: {len(results)}\n')

for i, r in enumerate(results[:5], 1):
    text = getattr(r.node, 'text', str(r))
    meta = getattr(r.node, 'metadata', {})

    # Method 1: Parse lineage from text header [Type | node_id | tenant]
    match = re.search(r'\[(\w+) \| ([\w-]+) \| (\w+)\]', text)
    if match:
        ntype, node_id, tenant = match.group(1), match.group(2), match.group(3)
        label = f'__{ntype}__{tenant}__'
        original = gs_direct.execute_query(f"MATCH (n:`{label}`) WHERE n.id = '{node_id}' RETURN properties(n) as props LIMIT 1")
        print(f'{i}. [{ntype}] node_id={node_id}')
        if original:
            props = original[0].get('props', {})
            display = props.get('name', props.get('title', props.get('description', 'N/A')))
            print(f'   Neptune: {display}')
        else:
            print(f'   (not in Neptune)')
    else:
        # Method 2: Check Source nodes in lexical-graph
        sources = gs_direct.execute_query('MATCH (s:`__Source__`) WHERE s.node_type IS NOT NULL RETURN s.node_type as ntype, s.node_id as nid, s.tenant as t LIMIT 5')
        if sources and i == 1:
            print(f'{i}. [Source nodes available]:')  
            for s in sources[:3]:
                print(f'   {s}')
        else:
            # Method 3: Entity correlation fallback
            ec = meta.get('entity_contexts', {}).get('contexts', [])
            entities = [ent.get('entity', {}).get('value', '') for ctx in ec[:3] for ent in ctx.get('entities', [])]
            print(f'{i}. [entities: {", ".join(entities[:3])}]')
    print()

print('Lineage correlation complete')


## Summary

The hybrid pipeline:
1. **Document-graph** stores structured data as typed nodes in Neptune
2. **Lexical-graph** indexes the same data for semantic search (OpenSearch vectors)
3. **Lineage** embedded in text `[source_type | node_id | tenant]` survives chunking
4. **Correlation** extracts lineage from query results → fetches original node from Neptune

Same graph database (Neptune) holds both the document-graph nodes AND the lexical-graph structure.
